In [2]:
from typing import Tuple
from collections import deque
import textstat

from tree.tree import Tree
from tree.node import Node, TerminalNode
from adapters.SemanticPerturb import PromptPackage
from adapters.TerminalPerturb import TerminalPerturber, PositionPerturber
from similarity.cosine_similarity import similarity
from model.engine import LLMAdapter
from adapters.OAI_Embeddings import RobertaEmbedder
import utils.constants as constants

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 109.7 MB/s eta 0:00:00



[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip
[nltk_data] Downloading package cmudict to /homes/lst20/nltk_data...
[nltk_data]   Package cmudict is already up-to-date!
[nltk_data] Downloading package wordnet to /homes/lst20/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [5]:
import os

directory = "/vol/bitbucket/lst20/long_POPQA_treenodes/prefix/gemma3-12b_perturb/3_2_0/tree/"  # Replace with your directory path
file_paths = []

for filename in os.listdir(directory):
    if filename.endswith(".pkl"):
        file_path = os.path.join(directory, filename)
        file_paths.append(file_path)

# Now pkl_strings contains the contents of all .pkl files as strings
file_paths[:3], len(file_paths)

(['/vol/bitbucket/lst20/long_POPQA_treenodes/prefix/gemma3-12b_perturb/3_2_0/tree/3241.pkl',
  '/vol/bitbucket/lst20/long_POPQA_treenodes/prefix/gemma3-12b_perturb/3_2_0/tree/1514.pkl',
  '/vol/bitbucket/lst20/long_POPQA_treenodes/prefix/gemma3-12b_perturb/3_2_0/tree/8952.pkl'],
 105)

In [13]:
'''
each question, answer tree has new value:
- is terminal (node)
- terminal_preturb applied (list of string)
'''
term_perturber = PositionPerturber("middle")

def generate_terminal_node(
    tree,
    parent_node : Node,
    use_parent_passage=True,) -> Tuple[TerminalNode, bool]:
    # take parent closest match
    if not use_parent_passage:
        # TODO: hehe
        raise NotImplementedError()
    new_state = {**parent_node.metadata}
    perturb_pkg = PromptPackage(text=parent_node.prompt, state=new_state)
    new_perturb_pkg: PromptPackage = term_perturber.terminal_perturb(perturb_pkg)
    # unpack results
    perturbation = new_perturb_pkg.text  # the new prompt
    perturb_state = new_perturb_pkg.state  # metadata collected by perturbers

    is_valid = perturb_state.get("is_valid", True)
    if is_valid:
        perturb_embedding = tree.embed_model.encode(perturbation)
        parent_embedding = parent_node.embedding.to("cuda:0")
        root_embedding = tree.root.embedding.to("cuda:0")
        sem_sim = similarity(perturb_embedding, parent_embedding)
        root_sim = similarity(perturb_embedding, root_embedding)
        rag_closest_match = parent_node.rag_closest_match
        rag_entities = parent_node.rag_entities
        ner_entities = parent_node.ner_entities
        
        wiki_title = parent_node.wiki_title
        fk_score = textstat.flesch_kincaid_grade(perturbation)
        dc_score = textstat.dale_chall_readability_score(perturbation)
        complexity_score = (fk_score + dc_score) / 2
        term_node = TerminalNode(
                    perturbation,
                    sem_sim,
                    root_sim,
                    tree.embed_model.encode(perturbation),
                    rag_closest_match,
                    rag_entities,
                    ner_entities,
                    wiki_title=wiki_title,
                    parent=parent_node,
                    fk_score=fk_score,
                    dc_score=dc_score,
                    complexity_score=complexity_score,
                )
        term_node.metadata.update(perturb_state)
    else:
        term_node = None
    print(term_node.metadata)
    return (term_node, is_valid)


def apply_terminal_preturb(tree: Tree):
    type_name = term_perturber.name
    root = tree.root
    level = 0
    queue = deque([(root, level)])
    
    while queue:  # BFS
        node, curr_level = queue.popleft()
        if isinstance(node, TerminalNode):
            continue  # Skip terminal nodes
        elif isinstance(node, Node) and type_name in node.metadata.get("terminal_name", []):
            continue  # Skip processed nodes
        elif isinstance(node, Node):
            # Generate a terminal node based on the current node
            term, is_valid = generate_terminal_node(tree, node, use_parent_passage=True)
            if is_valid:
                # Update node's metadata to include the type_name
                type_names = node.metadata.get("terminal_name", [])
                if not isinstance(type_names, list):
                    type_names = [type_names]
                if type_name not in type_names:
                    type_names.append(type_name)

                node.metadata["terminal_name"] = type_names

                # Set the metadata for the terminal node
                term.metadata = {
                    **term.metadata,
                    "level": curr_level
                }
            # Add children of the current node to the queue
            for child in node.children:
                queue.append((child, curr_level + 1))
            
            if is_valid: # Append the terminal node as a child
                node.add_child(term)

def process_tree(tree: Tree, tree_idx, model_name):
    tree.run_check_pop_qa_batched(tree_idx, model_name)

def get_generator(modelId: str) -> LLMAdapter:
    model: LLMAdapter = None
    if "gemma-3" in modelId.lower():
        from model.engine import Gemma3Adapter
        model = Gemma3Adapter(modelId)
    elif "gemma" in modelId.lower():
        from model.engine import GemmaAdapter
        model = GemmaAdapter(modelId)
    elif "mistralai" in modelId.lower():
        from model.engine import MistralInstructAdapter
        model = MistralInstructAdapter(modelId)
    elif "mistral.mistral-7b-instruct-v0:2" in modelId.lower():
        from model.engine import MistralInstructAwsAdapter
        model = MistralInstructAwsAdapter(modelId)
    else:
        raise NotImplementedError(f"No adapter implemented for: {modelId}")
    return model

In [14]:
def replicate_terminal_node_to_checked(question_tree, checked_trees: list):
    # parllel bfs to see missing nodes
    return question_tree, checked_trees
    # usage: update question_tree -> update existing 

In [5]:
gen_modelId = "google/gemma-3-1b-it"
embedder = RobertaEmbedder()
generator = get_generator(gen_modelId)
device = generator.model.device

In [ ]:
tree_id = 5474
dataset_name = "POPQA"
statrgy_path = "prefix"
inter_path = f"{constants.TREE_DIR}{dataset_name}_treenodes/{statrgy_path}/gemma3-12b_perturb/3_2_0/tree/{tree_id}.pkl"
final_path = f"{constants.TREE_DIR}{dataset_name}_treenodes/{statrgy_path}/gemma3-12b_perturb/3_2_0/{gen_modelId.replace('/', '-')}/complete/{tree_id}_checked.pkl"
terminal_tree_path = f"{constants.TREE_DIR}{dataset_name}_treenodes/{statrgy_path}/gemma3-12b_perturb/3_2_0/{gen_modelId.replace('/', '-')}/complete/terminal/{tree_id}_checked.pkl"

# inter_tree = Tree.load_tree(device, inter_path, embedder=embedder, eval="term")
full_tree = Tree.load_tree(device, final_path, embedder=embedder, generator=generator, eval='eval')
# add term to final tree
apply_terminal_preturb(full_tree)

{'terminal_name': 'question_position_middle', 'terminal_prompt': 'Who was the producer of Click?', 'is_valid': True, 'position': 'middle', 'num_extra_sent': 0}
{'root_prompt': 'Who was the producer of Click?', 'base_prompt': 'Who was the producer of Click?', 'prefix_similarity': -0.6777207255363464, 'is_valid': True, 'prefix_text': 'Dimensions: approx.', 'terminal_name': 'question_position_middle', 'terminal_prompt': 'Who was the producer of Click? Dimensions: approx.', 'position': 'middle', 'num_extra_sent': 1}
{'root_prompt': 'Who was the producer of Click?', 'base_prompt': 'Who was the producer of Click?', 'prefix_similarity': -0.6766003966331482, 'is_valid': True, 'prefix_text': 'The idea was pitched and sold in Q3 2002.', 'terminal_name': 'question_position_middle', 'terminal_prompt': 'Who was the producer of Click? The idea was pitched and sold in Q3 2002.', 'position': 'middle', 'num_extra_sent': 1}
{'root_prompt': 'Who was the producer of Click?', 'base_prompt': 'Who was the pr

In [9]:
from collections import deque

# Breadth-First Search traversal
def bfs_print(root, possible_answer=None):
    queue = deque([root])
    
    while queue:
        node = queue.popleft()
        
        # Print required information
        print("Prompt:", getattr(node, "prompt", None))
        print("Metadata:", getattr(node, "metadata", None))
        if node.answers:
            print(f"answers: {node.answers}")
            print(f"possible_answer: {possible_answer}")
            print(f"title: {[m[0]['title'] for m in root.rag_closest_match]}")
        print("Type:", type(node))
        print("------")
        
        # Add children if they exist
        children = getattr(node, "children", None)
        if children:
            queue.extend(children)


In [20]:
# # Start BFS from root
root = full_tree.root
bfs_print(root)

Prompt: Who was the producer of Click?
Metadata: {'terminal_name': ['question_position_middle']}
answers: {'google/gemma-3-1b-it': {'base': 'Steven Spielberg', 'base_rag': 'The producer of Click was Frank Coraci.'}}
possible_answer: None
title: ['Click (2006 film)', 'Click Click Boom', 'DoubleClick', 'Henrietta Lacks']
Type: <class 'tree.node.RootNode'>
------
Prompt: Dimensions: approx. Who was the producer of Click?
Metadata: {'root_prompt': 'Who was the producer of Click?', 'base_prompt': 'Who was the producer of Click?', 'prefix_similarity': -0.6777207255363464, 'is_valid': True, 'prefix_text': 'Dimensions: approx.', 'terminal_name': ['question_position_middle']}
answers: {'google/gemma-3-1b-it': {'base': 'Steven Spielberg.', 'base_rag': 'The producer of Click was Frank Coraci.'}}
possible_answer: None
title: ['Click (2006 film)', 'Click Click Boom', 'DoubleClick', 'Henrietta Lacks']
Type: <class 'tree.node.SemanticNode'>
------
Prompt: The idea was pitched and sold in Q3 2002. Who

In [2]:
from tree.tree import ReadTree
import utils.constants as constants

In [14]:
inter_dir = f'{constants.TREE_DIR}POPQA_treenodes/para/gemma3-12b_perturb/3_2_0/tree/'
tree_id = 11581
inter_path = f"{inter_dir}{tree_id}.pkl"
full_tree =  ReadTree.load_read_tree(inter_path)
# Start BFS from root
root = full_tree.root
bfs_print(root, full_tree.possible_answers)

Prompt: What is the capital of Province of Florence?
Metadata: {'terminal_applied': ['sg_dialect']}
Type: <class 'tree.node.RootNode'>
------
Prompt: What city serves as the main administrative center of the Florence province?
Metadata: {'root_prompt': 'What is the capital of Province of Florence?', 'base_prompt': 'What city serves as the main administrative center of the Florence province?', 'similarity': 0.8955578207969666, 'is_valid': True, 'terminal_applied': ['sg_dialect']}
Type: <class 'tree.node.SemanticNode'>
------
Prompt: Which city functions as the seat of government for the Florence province?
Metadata: {'root_prompt': 'What is the capital of Province of Florence?', 'base_prompt': 'Which city functions as the seat of government for the Florence province?', 'similarity': 0.8682377934455872, 'is_valid': True, 'terminal_applied': ['sg_dialect']}
Type: <class 'tree.node.SemanticNode'>
------
Prompt: What the capital of Province of Florence one?
Metadata: {'terminal_name': 'sg_di

In [17]:
inter_dir = f'{constants.TREE_DIR}POPQA_treenodes/para/gemma3-12b_perturb/3_2_0/google-gemma-3-12b-it/complete/'
tree_id = 4564
inter_path = f"{inter_dir}{tree_id}_checked.pkl"
full_tree =  ReadTree.load_read_tree(inter_path)
# Start BFS from root
root = full_tree.root
bfs_print(root, full_tree.possible_answers)

Prompt: Who was the producer of Rejected?
Metadata: {'terminal_applied': ['sg_dialect']}
answers: {'google/gemma-3-12b-it': {'base': 'Jay Johnston.', 'base_rag': 'Don Hertzfeldt.'}}
possible_answer: ["Don Hertzfeldt", "Donald Hertzfeldt"]
title: ['Rejected', 'Rejected takeoff', 'Cockney Rejects']
Type: <class 'tree.node.RootNode'>
------
Prompt: Who handled the production of Rejected?
Metadata: {'root_prompt': 'Who was the producer of Rejected?', 'base_prompt': 'Who handled the production of Rejected?', 'similarity': 0.9492934346199036, 'is_valid': True, 'terminal_applied': ['sg_dialect']}
answers: {'google/gemma-3-12b-it': {'base': 'James Belangia.', 'base_rag': 'Don Hertzfeldt. He photographed it on a 35mm rostrum camera he purchased in 1999.'}}
possible_answer: ["Don Hertzfeldt", "Donald Hertzfeldt"]
title: ['Rejected', 'Rejected takeoff', 'Cockney Rejects']
Type: <class 'tree.node.SemanticNode'>
------
Prompt: Who was responsible for making Rejected?
Metadata: {'root_prompt': 'Who 

In [42]:
# process final tree
process_tree(full_tree, tree_id, gen_modelId)
# write new full tree to term dir
full_tree.save_tree(terminal_tree_path)

Time to run check:  58.13671827316284


In [15]:
import os
from tree.tree import ReadTree
import numpy as np

positions = ["suffix", "middle"]

models = [
    "google/gemma-3-12b-it",
    "google/gemma-3-1b-it",
    "mistralai/Mistral-7B-Instruct-v0.2",
]

tree_ids = []
inter_dir = "/vol/bitbucket/lst20/long_POPQA_treenodes/prefix/gemma3-12b_perturb/3_2_0/tree/"
file_ext = ".pkl"
for filename in os.listdir(inter_dir):
    if filename.endswith(file_ext):
        tree_ids.append(int(filename[: -len(file_ext)]))

np.median(tree_ids)

7398.0

In [10]:
full_tree_long =ReadTree.load_read_tree(f"/vol/bitbucket/lst20/long_POPQA_treenodes/prefix/gemma3-12b_perturb/3_2_0/tree/{tree_ids[0]}.pkl")

root = full_tree_long.root
bfs_print(root, full_tree_long.possible_answers)

Prompt: Who is the father of Melora Hardin?
Metadata: {'terminal_applied': ['question_position_suffix', 'question_position_middle']}
Type: <class 'tree.node.RootNode'>
------
Prompt: Symantec Security Response - ABC, by Symantec Who is the father of Melora Hardin?
Metadata: {'root_prompt': 'Who is the father of Melora Hardin?', 'base_prompt': 'Who is the father of Melora Hardin?', 'prefix_similarity': -0.4317077398300171, 'is_valid': True, 'prefix_text': 'Symantec Security Response - ABC, by Symantec', 'terminal_applied': ['question_position_suffix', 'question_position_middle']}
Type: <class 'tree.node.SemanticNode'>
------
Prompt: Who is the father of Melora Hardin?
Metadata: {'terminal_name': 'question_position_suffix', 'terminal_prompt': 'Who is the father of Melora Hardin?', 'is_valid': True, 'position': 'suffix', 'num_extra_sent': 0, 'level': 0}
Type: <class 'tree.node.TerminalNode'>
------
Prompt: Who is the father of Melora Hardin?
Metadata: {'terminal_applied': ['question_posit